CELL 0 — Imports & Load inputs

In [8]:
import pandas as pd
from pathlib import Path

from fa_scripts.io import load_all
from fa_scripts.preprocess import (
    standardize_schedule, explode_to_team_rows, build_match_level_dataset,
    train_val_test_split_time
)

pd.set_option("display.max_columns", 200)
ROOT = Path(".")
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

frames = load_all()
frames.keys()


[INFO] Loaded fbref_team_defense.csv: 2,744 rows × 30 cols
[INFO] Loaded fbref_team_goal_shot_creation.csv: 2,744 rows × 28 cols
[INFO] Loaded fbref_team_match_stats.csv: 2,744 rows × 189 cols
[INFO] Loaded fbref_team_misc.csv: 2,744 rows × 30 cols
[INFO] Loaded fbref_team_passing.csv: 2,744 rows × 36 cols
[INFO] Loaded fbref_team_possession.csv: 2,744 rows × 37 cols
[INFO] Loaded fbref_team_schedule.csv: 2,744 rows × 23 cols
[INFO] Loaded fbref_team_shooting.csv: 2,744 rows × 29 cols
[INFO] Loaded understat_team_matches.csv: 1,372 rows × 27 cols


dict_keys(['fbref_team_defense', 'fbref_team_goal_shot_creation', 'fbref_team_match_stats', 'fbref_team_misc', 'fbref_team_passing', 'fbref_team_possession', 'fbref_team_schedule', 'fbref_team_shooting', 'understat_team_matches'])

CELL 1 — Build/confirm team_rows canonical table

In [2]:
# If you saved team_rows earlier, load it; else build from schedule
SNAPSHOT_DIR = PROC / "phase1_snapshot"
if (SNAPSHOT_DIR / "fbref_team_schedule.csv").exists() and "fbref_team_schedule" in frames:
    sched_src = frames["fbref_team_schedule"]
else:
    # fallback to Understat matches if schedule not available
    sched_src = frames["understat_team_matches"]

schedule_std = standardize_schedule(sched_src)
team_rows = explode_to_team_rows(schedule_std)
team_rows["date"] = pd.to_datetime(team_rows["date"], errors="coerce")
team_rows["team"] = team_rows["team"].astype("string")
team_rows["date_key"] = team_rows["date"].dt.normalize()

# Merge in Phase 1 metrics discovered during EDA (robust to missing xA)
# Example joins (adapt for your column names if different)
def try_merge(
    df,
    src,
    team_col_hints=("team", "squad", "home_team", "away_team", "home", "away"),
    date_col_hints=("match_date", "date"),
    cols_to_pick=None,
    rename_map=None,
):
    if src is None or src.empty:
        return df
    s = src.copy()
    s.columns = [c.lower() for c in s.columns]

    def pick(hints):
        for h in hints:
            h = h.lower()
            if h in s.columns:
                return h
        return None

    tcol = pick(team_col_hints)
    dcol = pick(date_col_hints)
    if tcol is None or dcol is None:
        return df

    s = s.rename(columns={tcol: "team"})
    s["team"] = s["team"].astype("string")
    s["date_key"] = pd.to_datetime(s[dcol], errors="coerce").dt.normalize()

    picks = []
    if cols_to_pick:
        for c in cols_to_pick:
            lc = c.lower()
            if lc in s.columns:
                picks.append(lc)
    keep = ["team", "date_key"] + picks
    s = s[keep].drop_duplicates(["team", "date_key"])

    out = df.merge(s, on=["team", "date_key"], how="left")
    if rename_map:
        rename_lc = {k.lower(): v for k, v in rename_map.items() if k.lower() in out.columns}
        out = out.rename(columns=rename_lc)
    return out

# Understat: xG, npxG, (xA if present)
ust = frames.get("understat_team_matches")
if ust is not None and not ust.empty:
    ust = ust.copy()
    ust.columns = [c.lower() for c in ust.columns]
    date_col = "match_date" if "match_date" in ust.columns else "date"
    ust["date_key"] = pd.to_datetime(ust[date_col], errors="coerce").dt.normalize()

    parts = []
    for side in ("home", "away"):
        team_col = f"{side}_team"
        if team_col not in ust.columns:
            continue
        part = pd.DataFrame({
            "team": ust[team_col].astype("string"),
            "date_key": ust["date_key"],
        })
        for alias, suffixes in {
            "xg": ["xg"],
            "npxg": ["np_xg"],
            "xa": ["xa", "x_a"],
        }.items():
            for suff in suffixes:
                col = f"{side}_{suff}"
                if col in ust.columns:
                    part[alias] = ust[col]
                    break
        parts.append(part)

    if parts:
        ust_long = pd.concat(parts, ignore_index=True)
        team_rows = team_rows.merge(ust_long, on=["team", "date_key"], how="left")

# FBref: possession & pass accuracy & discipline/defense
poss = frames.get("fbref_team_possession")
if poss is not None:
    team_rows = try_merge(
        team_rows,
        poss,
        cols_to_pick=["poss"],
        rename_map={"poss": "poss"}
    )

passing = frames.get("fbref_team_passing")
if passing is not None:
    pac_col = next((c for c in passing.columns if any(t in c.lower() for t in ["cmp%", "pass%", "completion"])), None)
    if pac_col:
        team_rows = try_merge(
            team_rows,
            passing,
            cols_to_pick=[pac_col],
            rename_map={pac_col: "pass_acc"}
        )

misc = frames.get("fbref_team_misc")
if misc is not None:
    y_col = next((c for c in misc.columns if "crdy" in c.lower() or "yellow" in c.lower()), None)
    r_col = next((c for c in misc.columns if "crdr" in c.lower() or "red" in c.lower()), None)
    picks = [c for c in [y_col, r_col] if c]
    if picks:
        rename_map = {}
        if y_col:
            rename_map[y_col] = "yellow_cards"
        if r_col:
            rename_map[r_col] = "red_cards"
        team_rows = try_merge(team_rows, misc, cols_to_pick=picks, rename_map=rename_map)

defe = frames.get("fbref_team_defense")
if defe is not None:
    clr_col = next((c for c in defe.columns if "clr" in c.lower() or "clear" in c.lower()), None)
    if clr_col:
        team_rows = try_merge(
            team_rows,
            defe,
            cols_to_pick=[clr_col],
            rename_map={clr_col: "clearances"}
        )

team_rows = team_rows.drop(columns=["date_key"], errors="ignore")
team_rows.head()


,date,home,away,home_goals,away_goals,season,team,opponent,is_home,gf,ga,result,match_id,xg,npxg,poss,pass_acc,yellow_cards,red_cards,clearances
0,2020-08-21 17:00:00,Bordeaux,Nantes,0,0,2020,Bordeaux,Nantes,1,0,0,draw,20200821_Bordeaux_Nantes,0.600075,0.600075,44.0,82.8,2.0,1.0,29.0
1,2020-08-22 15:00:00,Dijon,Angers,0,1,2020,Dijon,Angers,1,0,1,loss,20200822_Dijon_Angers,0.739709,0.739709,54.0,85.8,0.0,0.0,22.0
2,2020-08-22 19:00:00,Lille,Rennes,1,1,2020,Lille,Rennes,1,1,1,draw,20200822_Lille_Rennes,0.400974,0.400974,52.0,82.1,0.0,1.0,10.0
3,2020-08-23 13:00:00,Lorient,Strasbourg,3,1,2020,Lorient,Strasbourg,1,3,1,win,20200823_Lorient_Strasbourg,3.072140,2.312110,50.0,79.4,3.0,0.0,10.0
4,2020-08-23 11:00:00,Monaco,Reims,2,2,2020,Monaco,Reims,1,2,2,draw,20200823_Monaco_Reims,2.676220,2.676220,75.0,88.2,1.0,0.0,5.0


CELL 2 — Build the match-level dataset (+ rolling form & differentials)

In [4]:
# columns that are percentages in raw that we want as decimals
percent_cols = [c for c in ["poss","pass_acc"] if c in team_rows.columns]

# Build dataset (strict, no-leak rolling; pivot to home/away)
match_df = build_match_level_dataset(
    team_rows=team_rows,
    percent_cols=percent_cols,
    mean_metrics=[c for c in ["xg","npxg","poss","pass_acc"] if c in team_rows.columns],
    sum_metrics=[c for c in ["gf","ga","yellow_cards","red_cards","clearances"] if c in team_rows.columns],
    windows=(3,5,10),
    exp_alpha=0.8,
    keep_current_cols=[c for c in ["xg","npxg","poss","pass_acc","yellow_cards","red_cards","clearances"] if c in team_rows.columns],
    add_diff_for=None  # let the function pick sensible defaults
)

match_df.shape, match_df.columns[:20]


((1372, 106),
 Index(['match_id', 'date', 'season', 'home', 'away', 'home_goals',
        'home_conceded', 'home_xg', 'home_npxg', 'home_poss', 'home_pass_acc',
        'home_yellow_cards', 'home_red_cards', 'home_clearances', 'home_xg_l3',
        'home_xg_l5', 'home_xg_l10', 'home_xg_exp80', 'home_npxg_l3',
        'home_npxg_l5'],
       dtype='object'))

CELL 3 — Sanity checks & label view

In [5]:
# Label distribution (1 home win, 0 draw, -1 away win)
match_df["result"].value_counts(normalize=True).mul(100).round(1)

# Peek at engineered columns
[c for c in match_df.columns if c.endswith("_diff")][:20]


['home_xg_diff',
 'home_npxg_diff',
 'home_poss_diff',
 'home_pass_acc_diff',
 'home_xg_l3_diff',
 'home_xg_l5_diff',
 'home_xg_l10_diff',
 'home_xg_exp80_diff',
 'home_npxg_l3_diff',
 'home_npxg_l5_diff',
 'home_npxg_l10_diff',
 'home_npxg_exp80_diff',
 'home_poss_l3_diff',
 'home_poss_l5_diff',
 'home_poss_l10_diff',
 'home_poss_exp80_diff',
 'home_pass_acc_l3_diff',
 'home_pass_acc_l5_diff',
 'home_pass_acc_l10_diff',
 'home_pass_acc_exp80_diff']

CELL 4 — Chronological split (example: Train 2021–2023, Val 2024, Test 2025)

In [9]:
# Ensure season is string
match_df["season"] = match_df["season"].astype("string")

train_seasons = ["2021", "2022", "2023"]
val_seasons   = ["2024"]
test_seasons  = ["2025"]

train_df, val_df, test_df = train_val_test_split_time(match_df, train_seasons, val_seasons, test_seasons)

len(train_df), len(val_df), len(test_df)


(686, 306, 0)

CELL 5 — Save processed artifacts

In [12]:
from pathlib import Path

ROOT = Path("..").resolve()          # parent of notebooks/
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
out_dir = PROC / "processed_features"
out_dir.mkdir(parents=True, exist_ok=True)
ROOT = Path("..").resolve()                     # project root (one level up from notebooks/)
MODELS = ROOT / "models"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA = DATA_PROCESSED / "processed_features"


match_df.to_csv(out_dir / "match_level_dataset.csv", index=False)
train_df.to_csv(out_dir / "train.csv", index=False)
val_df.to_csv(out_dir / "val.csv", index=False)
test_df.to_csv(out_dir / "test.csv", index=False)
team_rows.to_csv(DATA / "team_rows_snapshot.csv", index=False)

print("Saved:", out_dir)


Saved: E:\Football Analyst\Portfolio\Predictive-Models\match-outcome-ligue1\data\processed\processed_features
